In [ ]:
pip install flask flask_cors pyngrok

In [ ]:
import pandas as pd
import json
import time
from datetime import datetime
from flask import Flask, request, jsonify, Response, stream_with_context
from flask_cors import CORS
import warnings
warnings.filterwarnings('ignore')

print("🤖 MULTI-MODEL Q&A PLATFORM")

print("\n📦 LOADING KAGGLE MODELS...")

try:
    import kaggle_benchmarks as kbench
    print("Kaggle benchmarks module loaded")
    
    ALL_MODELS = list(kbench.llms.keys())
    print(f"Found {len(ALL_MODELS)} models")
    
    MODELS = {}
    for model_name in ALL_MODELS:
        try:
            MODELS[model_name] = kbench.llms[model_name]
            print(f"   ✅ Loaded: {model_name}")
        except Exception as e:
            print(f"   ⚠️ Could not load {model_name}: {e}")
    
    print(f"\n✅ Loaded {len(MODELS)} models successfully")
    KAGGLE_AVAILABLE = True
    
except ImportError as e:
    print(f"⚠️ Kaggle benchmarks not available: {e}")
    KAGGLE_AVAILABLE = False
    MODELS = {}

app = Flask(__name__)
CORS(app)

qa_history = []

@app.route('/ask_stream', methods=['POST', 'OPTIONS'])
def ask_stream():
    """Stream answers from ALL models, one by one"""
    if request.method == 'OPTIONS':
        return '', 200
    
    try:
        data = request.get_json()
        question = data.get('question', '')
        session_id = data.get('session_id', '')
        
        if not question:
            return jsonify({'error': 'No question provided'}), 400
        
        print(f"\n📝 Question: {question[:50]}...")
        print(f"🤖 Will get answers from {len(MODELS)} models, one by one")
        
        def generate():
            model_list = list(MODELS.keys())
            total_models = len(model_list)
            results = []
            
            for idx, model_name in enumerate(model_list):
                print(f"  [{idx+1}/{total_models}] Getting answer from {model_name}...")
                
                try:
                    start_time = time.time()
                    model = MODELS[model_name]
                    response = model.prompt(question)
                    generation_time = time.time() - start_time
                    
                    result = {
                        'success': True,
                        'model': model_name,
                        'answer': response,
                        'time': round(generation_time, 2),
                        'index': idx,
                        'total': total_models
                    }
                    
                    print(f"    ✅ Got answer in {generation_time:.2f}s")
                    
                    qa_history.append({
                        'timestamp': datetime.now().isoformat(),
                        'question': question,
                        'model': model_name,
                        'answer': response,
                        'time': generation_time,
                        'session_id': session_id
                    })
                    
                    yield f"data: {json.dumps(result)}\n\n"
                    
                except Exception as e:
                    print(f"    ❌ Error: {e}")
                    result = {
                        'success': False,
                        'model': model_name,
                        'error': str(e),
                        'index': idx,
                        'total': total_models
                    }
                    yield f"data: {json.dumps(result)}\n\n"
                
                time.sleep(0.5)
            
            yield f"data: {json.dumps({'complete': True, 'total': total_models})}\n\n"
        
        return Response(stream_with_context(generate()), 
                        mimetype='text/event-stream',
                        headers={
                            'Cache-Control': 'no-cache',
                            'X-Accel-Buffering': 'no'
                        })
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/feedback', methods=['POST', 'OPTIONS'])
def submit_feedback():
    """Store user feedback for answers"""
    if request.method == 'OPTIONS':
        return '', 200
    
    try:
        data = request.get_json()
        
        feedback_file = __DIR__ + '/feedback.csv'
        
        import os
        file_exists = os.path.exists(feedback_file)
        
        with open(feedback_file, 'a') as f:
            if not file_exists:
                f.write("timestamp,question,model,answer,is_correct,correction,user_agent\n")
            
            f.write(f"{datetime.now().isoformat()},"
                   f'"{data.get("question","")}",'
                   f'"{data.get("model","")}",'
                   f'"{data.get("answer","")[:500]}",'
                   f'{data.get("is_correct",0)},'
                   f'"{data.get("correction","")}",'
                   f'"{request.headers.get("User-Agent","")}"\n')
        
        print(f"📊 Feedback saved for model: {data.get('model')}")
        
        return jsonify({'success': True})
        
    except Exception as e:
        print(f"❌ Feedback error: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/models', methods=['GET', 'OPTIONS'])
def list_models():
    """List all available models"""
    return jsonify({
        'models': list(MODELS.keys()),
        'count': len(MODELS)
    })

@app.route('/health', methods=['GET', 'OPTIONS'])
def health():
    """Health check"""
    return jsonify({
        'status': 'healthy',
        'models_loaded': len(MODELS),
        'total_queries': len(qa_history),
        'timestamp': datetime.now().isoformat()
    })

@app.route('/export', methods=['GET', 'OPTIONS'])
def export():
    """Export Q&A history"""
    df = pd.DataFrame(qa_history)
    csv_data = df.to_csv(index=False)
    
    return csv_data, 200, {
        'Content-Type': 'text/csv',
        'Content-Disposition': f'attachment; filename=qa_export_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    }

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        'name': 'Multi-Model Q&A Platform',
        'endpoints': {
            'POST /ask_stream': 'Stream answers from all models one by one',
            'POST /feedback': 'Submit feedback for answers',
            'GET /models': 'List all models',
            'GET /export': 'Export Q&A history'
        }
    })


try:
    from pyngrok import ngrok
    ngrok.set_auth_token("YOUR_NGROK_API_KEY")
    public_url = ngrok.connect(5000)
    print(f"\n🔗 PUBLIC URL: {public_url}")
    print("   Copy this URL for your PHP app!")
except Exception as e:
    print(f"\n⚠️ Ngrok error: {e}")

print("🤖 MULTI-MODEL Q&A PLATFORM - READY")
print(f"🤖 Models Loaded: {len(MODELS)}")
print("\n🌐 Endpoint:")
print("   POST /ask_stream - Stream answers from all models")
print("   POST /feedback - Submit feedback")

app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)

In [ ]:
import pandas as pd
import json
import time
from datetime import datetime
from flask import Flask, request, jsonify, Response, stream_with_context
from flask_cors import CORS
import warnings
warnings.filterwarnings('ignore')

print("🤖 MULTI-MODEL Q&A PLATFORM - STREAMING")

print("\n📦 LOADING KAGGLE MODELS...")

try:
    import kaggle_benchmarks as kbench
    print("✅ Kaggle benchmarks module loaded")
    
    ALL_MODELS = list(kbench.llms.keys())
    print(f"📋 Found {len(ALL_MODELS)} models")
    
    MODELS = {}
    for model_name in ALL_MODELS:
        try:
            MODELS[model_name] = kbench.llms[model_name]
            print(f"   ✅ Loaded: {model_name}")
        except Exception as e:
            print(f"   ⚠️ Could not load {model_name}: {e}")
    
    print(f"\n✅ Loaded {len(MODELS)} models successfully")
    KAGGLE_AVAILABLE = True
    
except ImportError as e:
    print(f"⚠️ Kaggle benchmarks not available: {e}")
    KAGGLE_AVAILABLE = False
    MODELS = {}

app = Flask(__name__)
CORS(app)

qa_history = []

@app.route('/ask_stream', methods=['POST', 'OPTIONS'])
def ask_stream():
    """Stream answers from ALL models, one by one"""
    if request.method == 'OPTIONS':
        return '', 200
    
    try:
        data = request.get_json()
        question = data.get('question', '')
        session_id = data.get('session_id', '')
        
        if not question:
            return jsonify({'error': 'No question provided'}), 400
        
        print(f"\n📝 Question: {question[:100]}...")
        print(f"🤖 Will get answers from {len(MODELS)} models, one by one")
        
        def generate():
            model_list = list(MODELS.keys())
            total_models = len(model_list)
            
            for idx, model_name in enumerate(model_list):
                print(f"  [{idx+1}/{total_models}] Getting answer from {model_name}...")
                
                try:
                    start_time = time.time()
                    model = MODELS[model_name]
                    response = model.prompt(question)
                    generation_time = time.time() - start_time
                    
                    result = {
                        'success': True,
                        'model': model_name,
                        'answer': response,
                        'time': round(generation_time, 2),
                        'index': idx,
                        'total': total_models
                    }
                    
                    print(f"    ✅ Got answer in {generation_time:.2f}s")
                    
                    qa_history.append({
                        'timestamp': datetime.now().isoformat(),
                        'question': question,
                        'model': model_name,
                        'answer': response,
                        'time': generation_time,
                        'session_id': session_id
                    })
                    
                    yield f"data: {json.dumps(result)}\n\n"
                    
                except Exception as e:
                    print(f"    ❌ Error: {e}")
                    result = {
                        'success': False,
                        'model': model_name,
                        'error': str(e),
                        'index': idx,
                        'total': total_models
                    }
                    yield f"data: {json.dumps(result)}\n\n"
                
                time.sleep(0.3)
            
            yield f"data: {json.dumps({'complete': True, 'total': total_models})}\n\n"
        
        return Response(stream_with_context(generate()), 
                        mimetype='text/event-stream',
                        headers={
                            'Cache-Control': 'no-cache',
                            'X-Accel-Buffering': 'no'
                        })
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/feedback', methods=['POST', 'OPTIONS'])
def submit_feedback():
    """Store user feedback for answers"""
    if request.method == 'OPTIONS':
        return '', 200
    
    try:
        data = request.get_json()
        
        feedback_file = __DIR__ + '/feedback.csv'
        
        import os
        file_exists = os.path.exists(feedback_file)
        
        with open(feedback_file, 'a') as f:
            if not file_exists:
                f.write("timestamp,question,model,answer,rating,confidence,note,user_agent\n")
            
            f.write(f"{datetime.now().isoformat()},"
                   f'"{data.get("question","")}",'
                   f'"{data.get("model","")}",'
                   f'"{data.get("answer","")[:1000]}",'
                   f'"{data.get("rating","")}",'
                   f'{data.get("confidence",3)},'
                   f'"{data.get("note","")}",'
                   f'"{request.headers.get("User-Agent","")}"\n')
        
        print(f"📊 Feedback saved for model: {data.get('model')}")
        
        return jsonify({'success': True})
        
    except Exception as e:
        print(f"❌ Feedback error: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/models', methods=['GET', 'OPTIONS'])
def list_models():
    """List all available models"""
    model_list = list(MODELS.keys())
    model_details = {}
    for m in model_list:
        provider = m.split('/')[0] if '/' in m else 'unknown'
        name = m.split('/')[-1] if '/' in m else m
        model_details[m] = {
            'name': name,
            'provider': provider,
            'full_name': m
        }
    
    return jsonify({
        'models': model_list,
        'count': len(MODELS),
        'details': model_details
    })

@app.route('/health', methods=['GET', 'OPTIONS'])
def health():
    """Health check"""
    return jsonify({
        'status': 'healthy',
        'models_loaded': len(MODELS),
        'total_queries': len(qa_history),
        'timestamp': datetime.now().isoformat()
    })

@app.route('/export', methods=['GET', 'OPTIONS'])
def export():
    """Export Q&A history"""
    df = pd.DataFrame(qa_history)
    csv_data = df.to_csv(index=False)
    
    return csv_data, 200, {
        'Content-Type': 'text/csv',
        'Content-Disposition': f'attachment; filename=qa_export_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    }

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        'name': 'Multi-Model Q&A Platform',
        'endpoints': {
            'POST /ask_stream': 'Stream answers from all models one by one',
            'POST /feedback': 'Submit feedback for answers',
            'GET /models': 'List all models'
        }
    })


try:
    from pyngrok import ngrok
    ngrok.set_auth_token("YOUR_NGROK_API_KEY")
    public_url = ngrok.connect(5000)
    print(f"\n🔗 PUBLIC URL: {public_url}")
    print("   Copy this URL for your PHP app!")
except Exception as e:
    print(f"\n⚠️ Ngrok error: {e}")

print("🤖 MULTI-MODEL Q&A PLATFORM - READY")
print(f"🤖 Models Loaded: {len(MODELS)}")
print("\n🌐 Endpoint:")
print("   POST /ask_stream - Stream answers from all models")
print("   POST /feedback - Submit feedback")

app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)